## Import Packages and Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer


### Helper Methods

In [ ]:
def smape(y_true, y_pred):
  denominator = (np.abs(y_true) + np.abs(y_pred)) / 200.0
  diff = np.abs(y_true - y_pred) / denominator
  diff[denominator == 0] = 0.0
  return np.nanmean(diff)

def visualize_predictions(y_train, y_test, y_pred):
  plt.figure(figsize=(6,4))
  plt.scatter(y_train.index, y_train, s=1, label='Train Data', color='green')
  plt.scatter(y_test.index, y_test, s=1, label='Actual', color='red', alpha=1)
  plt.scatter(y_test.index, y_pred, s=2, label='Actual', color='blue', alpha=.7)
  plt.show()

## Importing Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground September 2022/train.csv')

df['date'] = pd.to_datetime(df['date'])
df.set_index('row_id', inplace=True)

df.head()



## Visualizing Data

In [ ]:
figure, ax = plt.subplots(figsize=(17, 6))
by_country = df.groupby(['date', 'product',]).sum().unstack()
labels = by_country.columns
ax.plot(by_country)


ax.set_title('Cumulative Book Sales Over Time by Product')
plt.legend(by_country['num_sold'].columns)
plt.show()

## Feature Engineering

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground September 2022/train.csv')
df['date'] = pd.to_datetime(df['date'])
df.set_index('row_id', inplace=True)

country, store, product = 'Belgium', 'KaggleMart', 'Kaggle Advanced Techniques'

product_combination_index = (df['country']==country) & (df['store']==store) & (df['product']==product)
X = df.loc[product_combination_index, df.columns.drop('num_sold')].copy()
y = df.loc[product_combination_index, 'num_sold']

In [ ]:
def add_features(X_input):
  X = X_input.copy()
  X[['Year', 'Month', 'Day of Month']] = df['date'].astype(str).str.split('-', expand=True).astype(int)

  X['Day Of Week'] = X['date'].dt.dayofweek.values
  X['Day Of Year'] = X['date'].dt.dayofyear.values
  #X['Pandemic'] = (X['date'].dt.year.values >= 2020).astype(int)

  return X

def drop_unnecessary_features(X_input):
  X = X_input.copy()
  X = X.drop(['date', 'country', 'store', 'product'], axis=1)
  return X

X = add_features(X)
X = drop_unnecessary_features(X)


## Predictons

In [ ]:
X_train, X_test, y_train, y_test = X[:-365], X[-365:], y[:-365], y[-365:]

model = XGBRegressor(objective='reg:squarederror')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

visualize_predictions(y_train, y_test, y_pred)

pred_def = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

plt.figure(figsize=(6,4))
residuals = (pred_def['Predicted']-pred_def['Actual'])
sns.histplot(residuals)

plt.show()
print(smape(y_test, y_pred))

## Model Seperation for Each Product in Each Store in Each Country
- Each seperate combination of product has its own distinct pattern, so separating and traning each model for each unique combination is the most effective solution

Creating and Training All 48 Models

In [ ]:


df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground September 2022/train.csv')
df['date'] = pd.to_datetime(df['date'])
df.set_index('row_id', inplace=True)

combinations = []
for store in df['country'].unique():
  for country in df['store'].unique():
    for product in df['product'].unique():
      combinations.append([store, country, product])

best_models = {}
for combination in combinations: 
  country, store, product = combination[0], combination[1], combination[2]
  product_combination_index = (df['country']==country) & (df['store']==store) & (df['product']==product)
  X = df.loc[product_combination_index, df.columns.drop('num_sold')].copy()
  y = df.loc[product_combination_index, 'num_sold']
  adding_features_transformer = FunctionTransformer(add_features)
  dropping_features_transformer = FunctionTransformer(drop_unnecessary_features)

  steps = [("add_features", adding_features_transformer),
          ("dropping_unnecessary_features", dropping_features_transformer),
          ("xgb_model", XGBRegressor(objective='reg:squarederror'))] 

  # Standarized Scalar
  # Add Grid-Search to Model Here??? If not, take Pipeline out of Loop?

  X_train, X_test, y_train, y_test = X[:-365], X[-365:], y[:-365], y[-365:]

  xgb_pipeline = Pipeline(steps)
  xgb_pipeline.fit(X_train, y_train)
  best_models['-'.join(combination)] = xgb_pipeline
  print('-'.join(combination) + 'Model Found')




## Visualizing Combination Predictions

In [ ]:
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground September 2022/train.csv')
df['date'] = pd.to_datetime(df['date'])
df.set_index('row_id', inplace=True)

combinations = []
for store in df['country'].unique():
  for country in df['store'].unique():
    for product in df['product'].unique():
      combinations.append([store, country, product])

best_models = {}
for combination in combinations: 
  country, store, product = combination[0], combination[1], combination[2]
  product_combination_index = (df['country']==country) & (df['store']==store) & (df['product']==product)
  X = df.loc[product_combination_index, df.columns.drop('num_sold')].copy()
  y = df.loc[product_combination_index, 'num_sold']
  adding_features_transformer = FunctionTransformer(add_features)
  dropping_features_transformer = FunctionTransformer(drop_unnecessary_features)

  steps = [("add_features", adding_features_transformer),
          ("dropping_unnecessary_features", dropping_features_transformer),
          ('random_forest_model', RandomForestRegressor(n_estimators=5000))]
          #("xgb_model", XGBRegressor(objective='reg:squarederror'))] 

  # Add Grid-Search to Model Here??? If not, take Pipeline out of Loop?
  X_train, X_test, y_train, y_test = X[-365:], X[-365:], y[-365:], y[-365:] #X[:-365], X[-365:], y[:-365], y[-365:] 

  xgb_pipeline = Pipeline(steps)
  xgb_pipeline.fit(X_train, y_train) # Using only covid year
  best_models['-'.join(combination)] = xgb_pipeline
  print('-'.join(combination) + 'Model Found')

  y_pred = xgb_pipeline.predict(X_test)

  plt.figure(figsize=(6,4))
  plt.scatter(y_train.index, y_train, s=1, label='Train Data', color='green')
  plt.scatter(y_test.index, y_test, s=1, label='Actual', color='red', alpha=1)
  plt.scatter(y_test.index, y_pred, s=2, label='Actual', color='blue', alpha=.7)
  plt.show()



# Final Submissions

In [ ]:
X = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground September 2022/test.csv') # Elliot's Path
submission = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground September 2022/test.csv')[['row_id']]
X.head()

In [ ]:
X['date'] = pd.to_datetime(X['date'])
for combination in combinations: 
  country, store, product = combination[0], combination[1], combination[2]
  product_combination_index = (df['country']==country) & (df['store']==store) & (df['product']==product)

  combination_X = X.loc[product_combination_index, df.columns.drop('num_sold')].copy()
  predictions = best_models['-'.join(combination)].predict(combination_X)
  submission.loc[combination_X.index, 'num_sold'] = predictions

In [ ]:
submission

In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground September 2022/train.csv')
y = train_df['num_sold']

plt.figure(figsize=(6,4))
plt.scatter(y.index, y, s=1, label='Train Data', color='green')
plt.scatter(submission['row_id'], submission['num_sold'], s=2, label='Forecaster', color='blue', alpha=.7)
plt.show()

In [ ]:
submission.to_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground September 2022/submission.csv', index=False)